# SurfaceTarget-CRC — phase 04 — patch 4 4

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 4 — PATCH 4.4
#
# HGNC GENE IDENTIFIER CROSS-VALIDATION
#
# Objectives:
# 1. Validate existing DepMap candidate mappings
# 2. Download official HGNC complete gene annotations
# 3. Match Ensembl IDs to HGNC records
# 4. Extract NCBI Gene (Entrez) identifiers
# 5. Cross-check DepMap gene-column identifiers
# 6. Validate existing exact-symbol matches
# 7. Investigate previously unmatched genes
# 8. Flag ambiguous or conflicting annotations
# 9. Preserve existing dependency statistics
# 10. Save mapping audits and provenance
# 11. Create a Phase 4 research checkpoint
#
# IMPORTANT:
# - No automatic replacement of disputed mappings
# - No guessing from gene aliases
# - No automatic candidate exclusions
# - No CRISPR statistics recalculated
# - No original research results overwritten
# ============================================================


# ------------------------------------------------------------
# 1. IMPORTS
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import pandas as pd
import numpy as np
import requests
import hashlib
import json
import re
import os


# ------------------------------------------------------------
# 2. PROJECT CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"

DEPMAP = ROOT / "data/raw/depmap/26Q3"

HGNC_RAW = ROOT / "data/raw/hgnc"

RESULTS = ROOT / "results/tables"

CHECKPOINTS = ROOT / "checkpoints"

for folder in [
    META,
    HGNC_RAW,
    RESULTS,
    CHECKPOINTS
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


MAPPING_PATH = (
    RESULTS /
    "depmap_26q3_candidate_gene_mapping.csv"
)

SUMMARY_PATH = (
    RESULTS /
    "depmap_26q3_refined_dependency_summary.csv"
)

CRISPR_PATH = (
    DEPMAP /
    "CRISPRGeneEffect.csv"
)

HGNC_PATH = (
    HGNC_RAW /
    "hgnc_complete_set.txt"
)


HGNC_URL = (
    "https://storage.googleapis.com/"
    "public-download-files/hgnc/tsv/tsv/"
    "hgnc_complete_set.txt"
)


LEADING_GENES = [
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "CTHRC1",
    "S100P",
    "KRT80",
    "CEMIP",
    "INHBA",
    "WNT2"
]


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 4 — PATCH 4.4")
print("HGNC GENE IDENTIFIER CROSS-VALIDATION")
print("=" * 65)


# ------------------------------------------------------------
# 3. VERIFY EXISTING RESEARCH FILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RESEARCH INPUTS")
print("-" * 65)


required_paths = [
    MAPPING_PATH,
    SUMMARY_PATH,
    CRISPR_PATH
]


for path in required_paths:

    if not path.exists():
        raise FileNotFoundError(path)


mapping = pd.read_csv(
    MAPPING_PATH,
    dtype=str,
    keep_default_na=False
)


summary = pd.read_csv(
    SUMMARY_PATH,
    dtype=str,
    keep_default_na=False
)


if len(mapping) != 948:
    raise RuntimeError(
        "Unexpected DepMap candidate count."
    )


if len(summary) != 948:
    raise RuntimeError(
        "Unexpected refined dependency summary."
    )


if mapping["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate original gene identifiers."
    )


required_mapping_columns = {
    "gene_id",
    "ensembl_stable_id",
    "gene_name",
    "depmap_mapping_status",
    "depmap_column",
    "depmap_entrez_id"
}


if not required_mapping_columns.issubset(
    mapping.columns
):

    raise RuntimeError(
        "Unexpected candidate mapping schema."
    )


mapping["ensembl_stable_id"] = (
    mapping["ensembl_stable_id"]
    .astype(str)
    .str.strip()
)


if mapping[
    "ensembl_stable_id"
].duplicated().any():

    raise RuntimeError(
        "Duplicate stable Ensembl identifiers."
    )


originally_matched = (
    mapping["depmap_mapping_status"]
    .eq("Unique exact gene-symbol match")
)


if int(originally_matched.sum()) != 930:

    raise RuntimeError(
        "Expected 930 original symbol matches."
    )


print("\nCandidate genes:", len(mapping))

print(
    "Previously symbol-matched genes:",
    int(originally_matched.sum())
)

print(
    "Previously unmatched genes:",
    int((~originally_matched).sum())
)

print("\nResearch input validation: PASSED")


# ------------------------------------------------------------
# 4. DOWNLOAD OFFICIAL HGNC DATA
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("ACQUIRING OFFICIAL HGNC GENE ANNOTATIONS")
print("-" * 65)


session = requests.Session()

retry_policy = Retry(
    total=4,
    backoff_factor=1,
    status_forcelist=[
        429, 500, 502, 503, 504
    ],
    allowed_methods=["GET"]
)

session.mount(
    "https://",
    HTTPAdapter(
        max_retries=retry_policy
    )
)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


if HGNC_PATH.exists():

    print("\nExisting HGNC file found.")

    print("Reusing existing file.")

    download_status = "Reused"

else:

    print("\nDownloading HGNC complete dataset...")

    temporary = Path(
        str(HGNC_PATH) + ".part"
    )

    with session.get(
        HGNC_URL,
        stream=True,
        timeout=(30, 120)
    ) as response:

        response.raise_for_status()

        total_bytes = 0

        with open(
            temporary,
            "wb"
        ) as output:

            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):

                if not chunk:
                    continue

                output.write(chunk)

                total_bytes += len(chunk)

                if total_bytes > 50_000_000:

                    raise RuntimeError(
                        "HGNC download exceeded "
                        "the configured safety limit."
                    )

    os.replace(
        temporary,
        HGNC_PATH
    )

    download_status = "Downloaded"


hgnc_checksum = sha256_file(
    HGNC_PATH
)


print("\nHGNC acquisition:", download_status)

print(
    "Source size (MiB):",
    round(
        HGNC_PATH.stat().st_size / 1024**2,
        2
    )
)

print("SHA256:", hgnc_checksum)


# ------------------------------------------------------------
# 5. LOAD HGNC GENE ANNOTATIONS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING HGNC GENE ANNOTATIONS")
print("-" * 65)


hgnc = pd.read_csv(
    HGNC_PATH,
    sep="\t",
    dtype=str,
    keep_default_na=False,
    low_memory=False
)


required_hgnc_columns = {
    "hgnc_id",
    "symbol",
    "status",
    "ensembl_gene_id",
    "entrez_id"
}


missing_columns = (
    required_hgnc_columns
    - set(hgnc.columns)
)


if missing_columns:

    raise RuntimeError(
        "HGNC schema missing columns: "
        f"{missing_columns}"
    )


print("\nHGNC records:", len(hgnc))

print("HGNC columns:", len(hgnc.columns))


if hgnc["hgnc_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate HGNC identifiers."
    )


# Preserve original values while
# normalizing identifier fields.


for column in [
    "hgnc_id",
    "symbol",
    "status",
    "ensembl_gene_id",
    "entrez_id"
]:

    hgnc[column] = (
        hgnc[column]
        .astype(str)
        .str.strip()
    )


print("\nHGNC status distribution:")

print(
    hgnc["status"]
    .value_counts()
    .head(10)
    .to_string()
)


# ------------------------------------------------------------
# 6. PREPARE APPROVED ENSEMBL-TO-ENTREZ CROSSWALK
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING HGNC IDENTIFIER CROSSWALK")
print("-" * 65)


approved = hgnc[
    hgnc["status"].eq("Approved")
].copy()


approved = approved[
    approved["ensembl_gene_id"].ne("")
].copy()


approved["ensembl_gene_id"] = (
    approved["ensembl_gene_id"]
    .str.replace(
        r"\.\d+$",
        "",
        regex=True
    )
)


hgnc_by_ensembl = {

    gene_id: group.to_dict(
        orient="records"
    )

    for gene_id, group in approved.groupby(
        "ensembl_gene_id"
    )

}


print(
    "\nApproved HGNC records with Ensembl IDs:",
    len(approved)
)

print(
    "Unique Ensembl identifiers:",
    len(hgnc_by_ensembl)
)


# ------------------------------------------------------------
# 7. PARSE DEPMAP GENE IDENTIFIERS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PARSING DEPMAP GENE IDENTIFIERS")
print("-" * 65)


crispr_header = pd.read_csv(
    CRISPR_PATH,
    nrows=0
).columns.tolist()


if len(crispr_header) != 18435:

    raise RuntimeError(
        "Unexpected CRISPR gene header count."
    )


GENE_PATTERN = re.compile(
    r"^(.*?) \((\d+)\)$"
)


depmap_records = []

for column in crispr_header[1:]:

    match = GENE_PATTERN.fullmatch(
        column
    )

    if not match:

        raise RuntimeError(
            "Unexpected DepMap gene header: "
            f"{column}"
        )

    depmap_records.append({

        "depmap_column": column,

        "depmap_symbol":
            match.group(1),

        "depmap_entrez_id":
            match.group(2)
    })


depmap_genes = pd.DataFrame(
    depmap_records
)


depmap_by_entrez = {

    gene_id: group.to_dict(
        orient="records"
    )

    for gene_id, group in depmap_genes.groupby(
        "depmap_entrez_id"
    )

}


print("\nDepMap gene columns:", len(depmap_genes))

print(
    "Distinct NCBI Gene IDs:",
    len(depmap_by_entrez)
)


duplicate_entrez = (
    depmap_genes["depmap_entrez_id"]
    .duplicated()
    .sum()
)


print(
    "Repeated NCBI Gene IDs:",
    int(duplicate_entrez)
)


# ------------------------------------------------------------
# 8. CROSS-VALIDATE ALL 948 CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CROSS-VALIDATING ENSEMBL AND ENTREZ IDS")
print("-" * 65)


records = []


for _, row in mapping.iterrows():

    ensembl_id = row[
        "ensembl_stable_id"
    ]

    gene_name = row[
        "gene_name"
    ]

    prior_column = row[
        "depmap_column"
    ]

    prior_entrez = row[
        "depmap_entrez_id"
    ]

    prior_matched = (
        row["depmap_mapping_status"]
        == "Unique exact gene-symbol match"
    )


    hgnc_matches = hgnc_by_ensembl.get(
        ensembl_id,
        []
    )


    hgnc_symbol = ""

    hgnc_id = ""

    hgnc_entrez = ""

    proposed_column = ""

    proposed_depmap_symbol = ""

    status = ""


    # ----------------------------------------
    # No unambiguous HGNC Ensembl mapping
    # ----------------------------------------

    if len(hgnc_matches) == 0:

        status = (
            "No approved HGNC Ensembl mapping"
        )


    elif len(hgnc_matches) > 1:

        status = (
            "Ambiguous HGNC Ensembl mapping"
        )


    else:

        record = hgnc_matches[0]

        hgnc_symbol = record["symbol"]

        hgnc_id = record["hgnc_id"]

        hgnc_entrez = record[
            "entrez_id"
        ]


        # ------------------------------------
        # HGNC record lacks NCBI Gene ID
        # ------------------------------------

        if not hgnc_entrez:

            status = (
                "HGNC record lacks NCBI Gene ID"
            )


        else:

            depmap_matches = (
                depmap_by_entrez.get(
                    hgnc_entrez,
                    []
                )
            )


            # --------------------------------
            # No DepMap gene with this ID
            # --------------------------------

            if len(depmap_matches) == 0:

                status = (
                    "HGNC NCBI Gene ID absent "
                    "from DepMap"
                )


            # --------------------------------
            # Multiple DepMap gene columns
            # --------------------------------

            elif len(depmap_matches) > 1:

                status = (
                    "Ambiguous DepMap NCBI Gene ID"
                )


            else:

                depmap_match = depmap_matches[0]

                proposed_column = (
                    depmap_match["depmap_column"]
                )

                proposed_depmap_symbol = (
                    depmap_match["depmap_symbol"]
                )


                # ----------------------------
                # Existing symbol match agrees
                # with NCBI identifier
                # ----------------------------

                if (
                    prior_matched
                    and prior_entrez == hgnc_entrez
                    and prior_column == proposed_column
                ):

                    status = (
                        "Verified existing mapping"
                    )


                # ----------------------------
                # A previously unmatched gene
                # can now be located using ID
                # ----------------------------

                elif not prior_matched:

                    status = (
                        "Newly resolved by NCBI Gene ID"
                    )


                # ----------------------------
                # Symbol match and NCBI IDs
                # disagree: manual review
                # ----------------------------

                else:

                    status = (
                        "CONFLICT - identifier disagreement"
                    )


    records.append({

        "gene_id":
            row["gene_id"],

        "ensembl_stable_id":
            ensembl_id,

        "gene_name":
            gene_name,

        "previous_mapping_status":
            row["depmap_mapping_status"],

        "previous_depmap_column":
            prior_column,

        "previous_depmap_entrez_id":
            prior_entrez,

        "hgnc_id":
            hgnc_id,

        "hgnc_approved_symbol":
            hgnc_symbol,

        "hgnc_entrez_id":
            hgnc_entrez,

        "proposed_depmap_column":
            proposed_column,

        "proposed_depmap_symbol":
            proposed_depmap_symbol,

        "identity_validation_status":
            status,

        "symbol_changed":
            bool(
                hgnc_symbol
                and hgnc_symbol != gene_name
            )
    })


crosswalk = pd.DataFrame(
    records
)


if len(crosswalk) != 948:

    raise RuntimeError(
        "Gene count changed during crosswalk."
    )


if crosswalk["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate candidate records "
        "after identifier validation."
    )


print("\nCross-validation status:")

print(
    crosswalk[
        "identity_validation_status"
    ]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 9. SEPARATE VERIFIED AND UNRESOLVED MAPPINGS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("REVIEWING MAPPING RESOLUTION")
print("-" * 65)


verified = crosswalk[
    crosswalk["identity_validation_status"]
    .eq("Verified existing mapping")
].copy()


newly_resolved = crosswalk[
    crosswalk["identity_validation_status"]
    .eq("Newly resolved by NCBI Gene ID")
].copy()


conflicts = crosswalk[
    crosswalk["identity_validation_status"]
    .str.contains(
        "CONFLICT",
        na=False
    )
].copy()


unresolved = crosswalk[
    ~crosswalk["identity_validation_status"]
    .isin([
        "Verified existing mapping",
        "Newly resolved by NCBI Gene ID"
    ])
].copy()


print("\nVerified existing mappings:", len(verified))

print(
    "Newly resolved mappings:",
    len(newly_resolved)
)

print(
    "Identifier disagreements:",
    len(conflicts)
)

print(
    "Remaining unresolved or "
    "unverified mappings:",
    len(unresolved)
)


# ------------------------------------------------------------
# 10. INVESTIGATE THE ORIGINAL 18 UNMATCHED GENES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("ORIGINAL UNMATCHED GENE REVIEW")
print("-" * 65)


original_unmatched = crosswalk[
    ~crosswalk["previous_mapping_status"]
    .eq("Unique exact gene-symbol match")
].copy()


if len(original_unmatched) != 18:

    raise RuntimeError(
        "Unexpected number of previously "
        "unmatched genes."
    )


print("\nOriginal unmatched genes:", len(original_unmatched))


print("\n")

print(
    original_unmatched[
        [
            "gene_name",
            "hgnc_approved_symbol",
            "hgnc_entrez_id",
            "proposed_depmap_symbol",
            "identity_validation_status"
        ]
    ].to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 11. REVIEW IDENTIFIER DISAGREEMENTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("IDENTIFIER CONFLICT REVIEW")
print("-" * 65)


if conflicts.empty:

    print(
        "\nNo disagreements detected "
        "between validated existing mappings "
        "and the available HGNC crosswalk."
    )

else:

    print("\n")

    print(
        conflicts[
            [
                "gene_name",
                "previous_depmap_entrez_id",
                "hgnc_entrez_id",
                "previous_depmap_column",
                "proposed_depmap_column"
            ]
        ].to_string(
            index=False
        )
    )


print("\nIMPORTANT:")

print(
    "Conflicting mappings are not "
    "automatically replaced."
)

print(
    "Candidates without an HGNC match "
    "remain unverified, not absent."
)


# ------------------------------------------------------------
# 12. REVIEW LEADING CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING CANDIDATE IDENTITY AUDIT")
print("-" * 65)


leading = crosswalk[
    crosswalk["gene_name"]
    .isin(LEADING_GENES)
].copy()


leading["display_order"] = (
    leading["gene_name"]
    .map({
        gene: index
        for index, gene in enumerate(
            LEADING_GENES
        )
    })
)


leading = leading.sort_values(
    "display_order"
)


print("\n")

print(
    leading[
        [
            "gene_name",
            "hgnc_approved_symbol",
            "hgnc_entrez_id",
            "proposed_depmap_symbol",
            "identity_validation_status"
        ]
    ].to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 13. SAVE IDENTIFIER VALIDATION TABLES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING IDENTIFIER VALIDATION RESULTS")
print("-" * 65)


crosswalk.to_csv(

    RESULTS /
    "depmap_26q3_hgnc_crosswalk_audit.csv",

    index=False
)


verified.to_csv(

    RESULTS /
    "depmap_26q3_verified_gene_mapping.csv",

    index=False
)


newly_resolved.to_csv(

    RESULTS /
    "depmap_26q3_newly_resolved_genes.csv",

    index=False
)


conflicts.to_csv(

    RESULTS /
    "depmap_26q3_identifier_conflicts.csv",

    index=False
)


unresolved.to_csv(

    RESULTS /
    "depmap_26q3_unverified_gene_mapping.csv",

    index=False
)


original_unmatched.to_csv(

    RESULTS /
    "depmap_26q3_original_unmatched_review.csv",

    index=False
)


leading.to_csv(

    RESULTS /
    "depmap_26q3_leading_gene_identity_audit.csv",

    index=False
)


print("\nAll identifier audit tables saved.")


# ------------------------------------------------------------
# 14. SAVE RESEARCH PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING CROSSWALK PROVENANCE")
print("-" * 65)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "4.4",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "reference_source": (
        "HUGO Gene Nomenclature Committee"
    ),

    "reference_url": HGNC_URL,

    "reference_sha256": hgnc_checksum,

    "reference_local_path": str(HGNC_PATH),

    "reference_download_status": download_status,

    "reference_version_policy": (
        "HGNC complete dataset as retrieved "
        "at the recorded acquisition time"
    ),

    "depmap_release": "Public 26Q3",

    "candidate_count": int(
        len(crosswalk)
    ),

    "original_symbol_matched_genes": 930,

    "verified_existing_mappings": int(
        len(verified)
    ),

    "newly_resolved_mappings": int(
        len(newly_resolved)
    ),

    "identifier_conflicts": int(
        len(conflicts)
    ),

    "remaining_unverified_mappings": int(
        len(unresolved)
    ),

    "crosswalk_method": (
        "Ensembl stable gene ID to HGNC "
        "approved record to NCBI Gene ID "
        "to DepMap gene column"
    ),

    "ambiguous_mappings_automatically_selected":
        False,

    "original_gene_mapping_overwritten":
        False,

    "dependency_statistics_recalculated":
        False,

    "therapeutic_target_ranking_performed":
        False,

    "limitations": [

        "HGNC mappings reflect the database "
        "snapshot available at acquisition.",

        "Genes without HGNC NCBI IDs remain "
        "unverified rather than classified absent.",

        "Gene nomenclature can change between "
        "TCGA, HGNC, and DepMap releases.",

        "A verified identifier does not guarantee "
        "complete CRISPR measurement coverage.",

        "Disputed mappings require manual review.",

        "DepMap CRISPR gene effects are not "
        "direct measurements of antibody efficacy.",

        "No MSS status has been inferred.",

        "No therapeutic targets have been validated."

    ]
}


with open(

    META /
    "depmap_26q3_hgnc_crosswalk_provenance.json",

    "w"

) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


print("\nCrosswalk provenance saved.")


# ------------------------------------------------------------
# 15. CREATE PHASE 4 CHECKPOINT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING PHASE 4 RESEARCH CHECKPOINT")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S"
)


checkpoint = (

    CHECKPOINTS /

    f"SurfaceTarget_CRC_Phase4_Patch44_{timestamp}.zip"

)


# Preserve processed research outputs and
# metadata without duplicating the large
# raw DepMap and TCGA molecular datasets.
#
# The original DepMap CSV should also
# remain backed up on the user's computer.


backup_folders = [
    "data/metadata",
    "data/processed",
    "data/raw/cptac_colon",
    "data/raw/hpa",
    "data/raw/uniprot",
    "data/raw/hgnc",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]


with ZipFile(
    checkpoint,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        folder_path = ROOT / folder

        if not folder_path.exists():
            continue

        for file_path in folder_path.rglob("*"):

            if file_path.is_file():

                archive.write(

                    file_path,

                    arcname=str(
                        file_path.relative_to(ROOT)
                    )

                )


checkpoint_mib = (
    checkpoint.stat().st_size
    / (1024 ** 2)
)


print("\nCheckpoint created successfully.")

print(
    "Checkpoint size:",
    f"{checkpoint_mib:.2f} MiB"
)

print("Checkpoint location:")
print(checkpoint)


# ------------------------------------------------------------
# 16. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 4.4 SUMMARY")
print("=" * 65)


print("\nCandidate genes:", len(crosswalk))


print(
    "Original exact-symbol matches:",
    int(originally_matched.sum())
)


print(
    "Verified existing mappings:",
    len(verified)
)


print(
    "Newly resolved mappings:",
    len(newly_resolved)
)


print(
    "Identifier disagreements:",
    len(conflicts)
)


print(
    "Unresolved or unverified mappings:",
    len(unresolved)
)


print("\nSource: HGNC complete gene dataset")

print("HGNC source checksum: SAVED")

print("Crosswalk audit: SAVED")

print("Leading-gene identity review: SAVED")

print("Original 18-gene review: SAVED")

print("Research provenance: SAVED")

print("\nPhase 4 checkpoint: CREATED")

print(
    "Checkpoint size:",
    f"{checkpoint_mib:.2f} MiB"
)


print("\nNo dependency statistics recalculated.")

print("No existing gene mappings overwritten.")

print("No candidate genes excluded.")

print("No therapeutic target scores assigned.")


print("\nPATCH 4.4 COMPLETE")

print("=" * 65)